# 🛒 ShopEase E-Commerce Sales Database
## Celebal Summer Internship 2026 — Week 2 SQL Task
**Role:** Junior Data Analyst | **RDBMS:** SQLite (in-memory, standard SQL syntax)

---
### 📋 Objective
Analyze sales data using SQL with filtering, aggregation, joins, and business queries across a 4-table relational schema.

**Tables:** `customers` · `products` · `orders` · `order_items`

## ⚙️ Part 0 — Database Setup
Create schema, define constraints & indexes, then load all sample data.

In [1]:
import sqlite3
import pandas as pd

# Use in-memory SQLite — swap connection string for MySQL/PostgreSQL in production
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")   # enforce FK constraints
cur = con.cursor()

# ── Schema ─────────────────────────────────────────────────────────────────
cur.executescript("""
-- customers
CREATE TABLE customers (
    customer_id   INTEGER PRIMARY KEY,
    first_name    TEXT    NOT NULL,
    last_name     TEXT    NOT NULL,
    email         TEXT    UNIQUE NOT NULL,
    city          TEXT    NOT NULL,
    state         TEXT    NOT NULL,
    join_date     TEXT    NOT NULL,
    is_premium    INTEGER DEFAULT 0
);
CREATE INDEX idx_customers_city  ON customers(city);
CREATE INDEX idx_customers_state ON customers(state);

-- products
CREATE TABLE products (
    product_id    INTEGER PRIMARY KEY,
    product_name  TEXT    NOT NULL,
    category      TEXT    NOT NULL,
    brand         TEXT    NOT NULL,
    unit_price    REAL    NOT NULL CHECK (unit_price > 0),
    stock_qty     INTEGER NOT NULL DEFAULT 0 CHECK (stock_qty >= 0)
);
CREATE INDEX idx_products_category ON products(category);

-- orders
CREATE TABLE orders (
    order_id      INTEGER PRIMARY KEY,
    customer_id   INTEGER NOT NULL,
    order_date    TEXT    NOT NULL,
    status        TEXT    NOT NULL DEFAULT 'Pending'
                  CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')),
    total_amount  REAL    NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);
CREATE INDEX idx_orders_date   ON orders(order_date);
CREATE INDEX idx_orders_status ON orders(status);

-- order_items
CREATE TABLE order_items (
    item_id       INTEGER PRIMARY KEY,
    order_id      INTEGER NOT NULL,
    product_id    INTEGER NOT NULL,
    quantity      INTEGER NOT NULL CHECK (quantity > 0),
    unit_price    REAL    NOT NULL CHECK (unit_price > 0),
    discount_pct  REAL    DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100),
    FOREIGN KEY (order_id)   REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

print("✅ Schema created successfully.")

✅ Schema created successfully.


In [2]:
# ── Load sample data ───────────────────────────────────────────────────────
cur.executescript("""
INSERT INTO customers VALUES
(101,'Aarav','Sharma','aarav.s@email.com','Mumbai','Maharashtra','2024-01-15',1),
(102,'Priya','Patel','priya.p@email.com','Ahmedabad','Gujarat','2024-02-20',0),
(103,'Rohan','Gupta','rohan.g@email.com','Delhi','Delhi','2024-03-10',1),
(104,'Sneha','Reddy','sneha.r@email.com','Hyderabad','Telangana','2024-04-05',0),
(105,'Vikram','Singh','vikram.s@email.com','Jaipur','Rajasthan','2024-05-12',1),
(106,'Ananya','Iyer','ananya.i@email.com','Chennai','Tamil Nadu','2024-06-18',0),
(107,'Karan','Mehta','karan.m@email.com','Pune','Maharashtra','2024-07-22',1),
(108,'Divya','Nair','divya.n@email.com','Kochi','Kerala','2024-08-30',0);

INSERT INTO products VALUES
(201,'Wireless Earbuds','Electronics','BoAt',1499.00,250),
(202,'Cotton T-Shirt','Clothing','Levis',799.00,500),
(203,'Smart Watch','Electronics','Noise',2999.00,150),
(204,'Running Shoes','Clothing','Nike',4599.00,120),
(205,'Bluetooth Speaker','Electronics','JBL',3499.00,200),
(206,'Bedsheet Set','Home','Spaces',1299.00,300),
(207,'Laptop Stand','Electronics','AmazonBasics',899.00,180),
(208,'Cushion Covers (Set)','Home','HomeCenter',599.00,400);

INSERT INTO orders VALUES
(1001,101,'2024-08-01','Delivered',4498.00),
(1002,102,'2024-08-03','Delivered',799.00),
(1003,103,'2024-08-05','Shipped',7498.00),
(1004,101,'2024-08-10','Delivered',3499.00),
(1005,104,'2024-08-12','Cancelled',2999.00),
(1006,105,'2024-08-15','Delivered',5898.00),
(1007,106,'2024-08-18','Pending',1299.00),
(1008,103,'2024-08-20','Delivered',899.00),
(1009,107,'2024-08-25','Shipped',6098.00),
(1010,108,'2024-08-28','Delivered',1598.00);

INSERT INTO order_items VALUES
(5001,1001,201,2,1499.00,0),(5002,1001,207,1,899.00,10),
(5003,1002,202,1,799.00,0),(5004,1003,203,1,2999.00,0),
(5005,1003,204,1,4599.00,5),(5006,1004,205,1,3499.00,0),
(5007,1005,203,1,2999.00,0),(5008,1006,201,1,1499.00,10),
(5009,1006,204,1,4599.00,5),(5010,1007,206,1,1299.00,0),
(5011,1008,207,1,899.00,0),(5012,1009,205,1,3499.00,0),
(5013,1009,208,2,599.00,15),(5014,1010,206,1,1299.00,0),
(5015,1010,208,1,599.00,0);
""")

print("✅ Sample data loaded successfully.")

# Helper to run SQL and display as DataFrame
def run(sql, label=""):
    df = pd.read_sql_query(sql, con)
    if label:
        print(f"▶ {label}")
    return df

✅ Sample data loaded successfully.


---
## 📘 Section A — SQL Basics
### SELECT, Constraints, Primary Keys

### Q1 — Display all columns and rows from the `customers` table

In [3]:
run("SELECT * FROM customers", "Q1: All customers")

▶ Q1: All customers


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


### Q2 — Retrieve `first_name`, `last_name`, and `city` of all customers

In [4]:
run("SELECT first_name, last_name, city FROM customers", "Q2: Name & city")

▶ Q2: Name & city


,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


### Q3 — List all unique categories in the `products` table

In [5]:
run("SELECT DISTINCT category FROM products", "Q3: Unique categories")

▶ Q3: Unique categories


,category
0,Clothing
1,Electronics
2,Home


### Q4 — Primary Keys in each table

| Table | Primary Key |
|---|---|
| customers | customer_id |
| products | product_id |
| orders | order_id |
| order_items | item_id |

**Why UNIQUE & NOT NULL?**
- **UNIQUE** — prevents two rows from representing the same entity. Two customers sharing an ID would make lookups ambiguous.
- **NOT NULL** — a NULL means "unknown." An unknown identifier cannot uniquely identify any row, breaking JOINs and referential integrity.

### Q5 — Constraints on the `email` column

The `email` column has two constraints:
1. **UNIQUE** — no two customers may share the same email.
2. **NOT NULL** — every customer must supply an email.

**On a duplicate insert:**
- MySQL → `ERROR 1062: Duplicate entry 'x@y.com' for key 'email'`
- PostgreSQL → `ERROR 23505: duplicate key value violates unique constraint`

The INSERT is rejected and no row is written.

### Q6 — Inserting `unit_price = -50` — Constraint Violation

```sql
INSERT INTO products (product_id, product_name, category, brand, unit_price, stock_qty)
VALUES (299, 'Test Product', 'Electronics', 'TestBrand', -50.00, 10);
```

The `CHECK (unit_price > 0)` constraint rejects any price ≤ 0.
- MySQL → `ERROR 3819: Check constraint 'products_chk_1' is violated`
- PostgreSQL → `ERROR 23514: violates check constraint "products_unit_price_check"`

In [6]:
# Demonstrate the constraint violation
try:
    con.execute("""
        INSERT INTO products (product_id, product_name, category, brand, unit_price, stock_qty)
        VALUES (299, 'Test Product', 'Electronics', 'TestBrand', -50.00, 10)
    """)
except Exception as e:
    print(f"❌ Constraint violation caught:\n   {e}")

❌ Constraint violation caught:
   CHECK constraint failed: unit_price > 0


---
## 🔍 Section B — Filtering & Optimization
### WHERE Clauses and Index Performance

### Q7 — Retrieve all orders with `status = 'Delivered'`

In [7]:
run("SELECT * FROM orders WHERE status = 'Delivered'", "Q7: Delivered orders")
# Insight: 6 out of 10 orders (60%) were successfully delivered

▶ Q7: Delivered orders


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498.0
1,1002,102,2024-08-03,Delivered,799.0
2,1004,101,2024-08-10,Delivered,3499.0
3,1006,105,2024-08-15,Delivered,5898.0
4,1008,103,2024-08-20,Delivered,899.0
5,1010,108,2024-08-28,Delivered,1598.0


### Q8 — Electronics products with `unit_price > ₹2000`

In [8]:
run("""
SELECT *
FROM   products
WHERE  category = 'Electronics'
  AND  unit_price > 2000
""", "Q8: Electronics > ₹2000")

▶ Q8: Electronics > ₹2000


,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999.0,150
1,205,Bluetooth Speaker,Electronics,JBL,3499.0,200


### Q9 — Customers from Maharashtra who joined in 2024

In [9]:
# SARGable date range keeps the query index-friendly
run("""
SELECT *
FROM   customers
WHERE  join_date >= '2024-01-01'
  AND  join_date <  '2025-01-01'
  AND  state = 'Maharashtra'
""", "Q9: Maharashtra customers, 2024")
# Insight: Both Maharashtra customers are premium members

▶ Q9: Maharashtra customers, 2024


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


### Q10 — Orders between Aug 10–25 that are NOT Cancelled

In [10]:
run("""
SELECT *
FROM   orders
WHERE  order_date BETWEEN '2024-08-10' AND '2024-08-25'
  AND  status <> 'Cancelled'
""", "Q10: Active orders Aug 10-25")
# Note: Order 1005 (Aug 12, Cancelled) is excluded

▶ Q10: Active orders Aug 10-25


,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499.0
1,1006,105,2024-08-15,Delivered,5898.0
2,1007,106,2024-08-18,Pending,1299.0
3,1008,103,2024-08-20,Delivered,899.0
4,1009,107,2024-08-25,Shipped,6098.0


### Q11 — How `idx_orders_date` improves query performance

`idx_orders_date` is a **B-Tree index** on `orders(order_date)`.

| Without Index | With Index |
|---|---|
| Full table scan — reads every row | Binary search on sorted B-Tree |
| I/O complexity: O(n) | I/O complexity: O(log n) |

The query below benefits directly from this index:

In [11]:
# This query gets an index range-seek on order_date instead of a full scan
run("""
SELECT *
FROM   orders
WHERE  order_date BETWEEN '2024-08-01' AND '2024-08-31'
""", "Q11: August orders (index-accelerated)")

▶ Q11: August orders (index-accelerated)


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498.0
1,1002,102,2024-08-03,Delivered,799.0
2,1003,103,2024-08-05,Shipped,7498.0
3,1004,101,2024-08-10,Delivered,3499.0
4,1005,104,2024-08-12,Cancelled,2999.0
5,1006,105,2024-08-15,Delivered,5898.0
6,1007,106,2024-08-18,Pending,1299.0
7,1008,103,2024-08-20,Delivered,899.0
8,1009,107,2024-08-25,Shipped,6098.0
9,1010,108,2024-08-28,Delivered,1598.0


### Q12 — SARGable rewrite: avoiding `YEAR()` on indexed columns

**Non-SARGable (index bypassed):**
```sql
SELECT * FROM customers WHERE YEAR(join_date) = 2024;
```
`YEAR()` wraps the column — the optimizer cannot use the index and performs a full scan.

**SARGable rewrite (index used):**

In [12]:
# Column is bare on the left — optimizer can use idx_customers_join_date
run("""
SELECT *
FROM   customers
WHERE  join_date >= '2024-01-01'
  AND  join_date <  '2025-01-01'
""", "Q12: SARGable date filter")

▶ Q12: SARGable date filter


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


---
## 📊 Section C — Aggregation
### GROUP BY, SUM, COUNT, AVG, MIN, MAX

### Q13 — Total number of orders

In [13]:
run("SELECT COUNT(*) AS total_orders FROM orders", "Q13: Total orders")

▶ Q13: Total orders


,total_orders
0,10


### Q14 — Total revenue from Delivered orders

In [14]:
run("""
SELECT SUM(total_amount) AS total_revenue
FROM   orders
WHERE  status = 'Delivered'
""", "Q14: Revenue (Delivered)")
# Insight: ₹17,191 delivered out of ₹35,085 gross (~48%)

▶ Q14: Revenue (Delivered)


,total_revenue
0,17191.0


### Q15 — Average `unit_price` per category

In [15]:
run("""
SELECT   category,
         ROUND(AVG(unit_price), 2) AS avg_price
FROM     products
GROUP BY category
""", "Q15: Avg price per category")
# Insight: Clothing avg is highest (₹2,699) due to Running Shoes (₹4,599)

▶ Q15: Avg price per category


,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


### Q16 — Order count and revenue by status, sorted by revenue

In [16]:
run("""
SELECT   status,
         COUNT(*)                    AS order_count,
         ROUND(SUM(total_amount), 2) AS total_revenue
FROM     orders
GROUP BY status
ORDER BY total_revenue DESC
""", "Q16: Revenue by status")
# Insight: ₹13,596 is still in-transit (Shipped status)

▶ Q16: Revenue by status


,status,order_count,total_revenue
0,Delivered,6,17191.0
1,Shipped,2,13596.0
2,Cancelled,1,2999.0
3,Pending,1,1299.0


### Q17 — Most & least expensive product per category

In [17]:
run("""
SELECT   category,
         MAX(unit_price) AS max_price,
         MIN(unit_price) AS min_price
FROM     products
GROUP BY category
""", "Q17: Max/Min price per category")

▶ Q17: Max/Min price per category


,category,max_price,min_price
0,Clothing,4599.0,799.0
1,Electronics,3499.0,899.0
2,Home,1299.0,599.0


### Q18 — Categories where average price > ₹2000 (HAVING clause)

In [18]:
run("""
SELECT   category,
         ROUND(AVG(unit_price), 2) AS avg_price
FROM     products
GROUP BY category
HAVING   AVG(unit_price) > 2000
""", "Q18: High-value categories (HAVING)")
# Insight: Home (avg ₹949) does not qualify — only Clothing & Electronics

▶ Q18: High-value categories (HAVING)


,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


---
## 🔗 Section D — Joins & Relationships

### Q19 — Orders with customer name (INNER JOIN)

In [19]:
run("""
SELECT  o.order_id,
        o.order_date,
        c.first_name,
        c.last_name,
        o.total_amount
FROM    orders   o
        INNER JOIN customers c ON o.customer_id = c.customer_id
""", "Q19: Orders + customer names")
# Insight: Top spender is Rohan Gupta — order 1003 worth ₹7,498

▶ Q19: Orders + customer names


,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498.0
1,1002,2024-08-03,Priya,Patel,799.0
2,1003,2024-08-05,Rohan,Gupta,7498.0
3,1004,2024-08-10,Aarav,Sharma,3499.0
4,1005,2024-08-12,Sneha,Reddy,2999.0
5,1006,2024-08-15,Vikram,Singh,5898.0
6,1007,2024-08-18,Ananya,Iyer,1299.0
7,1008,2024-08-20,Rohan,Gupta,899.0
8,1009,2024-08-25,Karan,Mehta,6098.0
9,1010,2024-08-28,Divya,Nair,1598.0


### Q20 — All customers and their orders (LEFT JOIN)
Customers with no orders would show NULL — guaranteed by LEFT JOIN.

In [20]:
run("""
SELECT  c.customer_id,
        c.first_name,
        c.last_name,
        o.order_id,
        o.total_amount
FROM    customers c
        LEFT JOIN orders o ON c.customer_id = o.customer_id
""", "Q20: All customers + orders (LEFT JOIN)")

▶ Q20: All customers + orders (LEFT JOIN)


,customer_id,first_name,last_name,order_id,total_amount
0,101,Aarav,Sharma,1004,3499.0
1,101,Aarav,Sharma,1001,4498.0
2,102,Priya,Patel,1002,799.0
3,103,Rohan,Gupta,1008,899.0
4,103,Rohan,Gupta,1003,7498.0
5,104,Sneha,Reddy,1005,2999.0
6,105,Vikram,Singh,1006,5898.0
7,106,Ananya,Iyer,1007,1299.0
8,107,Karan,Mehta,1009,6098.0
9,108,Divya,Nair,1010,1598.0


### Q21 — Order items with product details (3-table JOIN)

In [21]:
run("""
SELECT  oi.order_id,
        p.product_name,
        oi.quantity,
        oi.unit_price,
        oi.discount_pct
FROM    order_items oi
        JOIN orders   o ON oi.order_id   = o.order_id
        JOIN products p ON oi.product_id = p.product_id
""", "Q21: Line-item detail (3-table JOIN)")

▶ Q21: Line-item detail (3-table JOIN)


,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499.0,0.0
1,1001,Laptop Stand,1,899.0,10.0
2,1002,Cotton T-Shirt,1,799.0,0.0
3,1003,Smart Watch,1,2999.0,0.0
4,1003,Running Shoes,1,4599.0,5.0
5,1004,Bluetooth Speaker,1,3499.0,0.0
6,1005,Smart Watch,1,2999.0,0.0
7,1006,Wireless Earbuds,1,1499.0,10.0
8,1006,Running Shoes,1,4599.0,5.0
9,1007,Bedsheet Set,1,1299.0,0.0


### Q22 — LEFT JOIN vs RIGHT JOIN vs FULL OUTER JOIN

| Join Type | Returns |
|---|---|
| **LEFT JOIN** | All rows from LEFT table + matched right rows (unmatched → NULL) |
| **RIGHT JOIN** | All rows from RIGHT table + matched left rows (unmatched → NULL) |
| **FULL OUTER JOIN** | All rows from BOTH tables (unmatched on either side → NULL) |

**When to use FULL OUTER JOIN:** Reconciling two systems — e.g., find customers who never ordered *and* orders whose customer was deleted.

> ⚠️ MySQL does not support `FULL OUTER JOIN` natively. Emulate with `LEFT JOIN ... UNION ... RIGHT JOIN`.

### Q23 — Foreign Key Relationships

```
customers.customer_id  ◀── orders.customer_id
orders.order_id        ◀── order_items.order_id
products.product_id    ◀── order_items.product_id
```

**What happens with `customer_id = 999` (non-existent)?**
- MySQL → `ERROR 1452: Cannot add or update a child row: a foreign key constraint fails`
- PostgreSQL → `ERROR 23503: insert or update violates foreign key constraint`

The database enforces **referential integrity** — every order must reference a valid customer.

In [22]:
# Demonstrate FK violation
try:
    con.execute("""
        INSERT INTO orders (order_id, customer_id, order_date, status, total_amount)
        VALUES (1099, 999, '2024-09-01', 'Pending', 500.00)
    """)
except Exception as e:
    print(f"❌ FK constraint violation caught:\n   {e}")

❌ FK constraint violation caught:
   FOREIGN KEY constraint failed


---
## 🧠 Section E — Advanced Concepts
### CASE, ACID Properties, Transactions

### Q24 — Classify products into price tiers using CASE

In [23]:
run("""
SELECT  product_name,
        unit_price,
        CASE
            WHEN unit_price < 1000                THEN 'Budget'
            WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
            ELSE                                       'Premium'
        END AS price_tier
FROM    products
""", "Q24: Price tier classification")
# Insight: 3 Budget | 3 Mid-Range | 2 Premium

▶ Q24: Price tier classification


,product_name,unit_price,price_tier
0,Wireless Earbuds,1499.0,Mid-Range
1,Cotton T-Shirt,799.0,Budget
2,Smart Watch,2999.0,Mid-Range
3,Running Shoes,4599.0,Premium
4,Bluetooth Speaker,3499.0,Premium
5,Bedsheet Set,1299.0,Mid-Range
6,Laptop Stand,899.0,Budget
7,Cushion Covers (Set),599.0,Budget


### Q25 — Delivered vs Not-Delivered in a single row (CASE inside SUM)

In [24]:
run("""
SELECT
    SUM(CASE WHEN status = 'Delivered'  THEN 1 ELSE 0 END) AS delivered,
    SUM(CASE WHEN status <> 'Delivered' THEN 1 ELSE 0 END) AS not_delivered
FROM orders
""", "Q25: Delivery pivot")
# Insight: 60% fulfilment rate (6 delivered, 4 not)

▶ Q25: Delivery pivot


,delivered,not_delivered
0,6,4


### Q26 — ACID Properties Explained

| Letter | Property | Meaning |
|---|---|---|
| **A** | **Atomicity** | All operations succeed or all are rolled back — no partial transactions |
| **C** | **Consistency** | Transaction moves DB from one valid state to another, respecting all constraints |
| **I** | **Isolation** | Concurrent transactions are invisible to each other until committed |
| **D** | **Durability** | Committed changes survive crashes (written to disk via WAL/redo log) |

**Bank transfer example:**
- **A** → Debit ₹5,000 from A and credit ₹5,000 to B must both succeed, or neither does.
- **C** → Neither balance can go negative (CHECK constraint).
- **I** → Two tellers see consistent balances; neither can read the other's mid-transaction state.
- **D** → Once committed, the transfer persists even if the server crashes immediately after.

### Q27 — Full Transaction: New order + 2 items + stock update

In [25]:
try:
    con.execute("BEGIN")

    # Step 1: Insert new order
    con.execute("""
        INSERT INTO orders (order_id, customer_id, order_date, status, total_amount)
        VALUES (1011, 102, date('now'), 'Pending', 1598.00)
    """)

    # Step 2a: First order item — Bedsheet Set
    con.execute("""
        INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
        VALUES (5016, 1011, 206, 1, 1299.00, 0)
    """)

    # Step 2b: Second order item — Cushion Covers
    con.execute("""
        INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_pct)
        VALUES (5017, 1011, 208, 1, 599.00, 0)
    """)

    # Step 3a: Reduce stock for Bedsheet Set
    con.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206")

    # Step 3b: Reduce stock for Cushion Covers
    con.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208")

    con.execute("COMMIT")
    print("✅ Transaction committed successfully.")

except Exception as e:
    con.execute("ROLLBACK")
    print(f"❌ Transaction rolled back due to: {e}")

❌ Transaction rolled back due to: cannot start a transaction within a transaction


In [26]:
# Verify the transaction result
print("New order:")
display(run("SELECT * FROM orders WHERE order_id = 1011"))

print("\nNew order items:")
display(run("SELECT * FROM order_items WHERE order_id = 1011"))

print("\nUpdated stock (products 206 & 208):")
display(run("SELECT product_id, product_name, stock_qty FROM products WHERE product_id IN (206, 208)"))

New order:


,order_id,customer_id,order_date,status,total_amount



New order items:


,item_id,order_id,product_id,quantity,unit_price,discount_pct



Updated stock (products 206 & 208):


,product_id,product_name,stock_qty
0,206,Bedsheet Set,300
1,208,Cushion Covers (Set),400


---
## ✅ Validation — Data Quality Checks

In [27]:
print("=" * 55)
print("DATA QUALITY VALIDATION REPORT")
print("=" * 55)

checks = {
    "Row counts per table": """
        SELECT 'customers'  AS tbl, COUNT(*) AS rows FROM customers UNION ALL
        SELECT 'products',          COUNT(*)          FROM products  UNION ALL
        SELECT 'orders',            COUNT(*)          FROM orders    UNION ALL
        SELECT 'order_items',       COUNT(*)          FROM order_items
    """,
    "NULL emails": "SELECT COUNT(*) AS null_emails FROM customers WHERE email IS NULL",
    "Duplicate emails": """
        SELECT email, COUNT(*) AS cnt FROM customers
        GROUP BY email HAVING COUNT(*) > 1
    """,
    "Orphaned orders (no customer)": """
        SELECT o.order_id FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,
    "Orphaned order_items (no order)": """
        SELECT oi.item_id FROM order_items oi
        LEFT JOIN orders o ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,
    "Negative stock": """
        SELECT product_id, product_name, stock_qty
        FROM products WHERE stock_qty < 0
    """,
}

for label, sql in checks.items():
    df = pd.read_sql_query(sql, con)
    status = "✅" if len(df) <= 4 else "⚠️"  # row counts ≤ 4 tables is expected
    print(f"\n{status} {label}:")
    print(df.to_string(index=False))

DATA QUALITY VALIDATION REPORT

✅ Row counts per table:
        tbl  rows
  customers     8
   products     8
     orders    10
order_items    15

✅ NULL emails:
 null_emails
           0

✅ Duplicate emails:
Empty DataFrame
Columns: [email, cnt]
Index: []

✅ Orphaned orders (no customer):
Empty DataFrame
Columns: [order_id]
Index: []

✅ Orphaned order_items (no order):
Empty DataFrame
Columns: [item_id]
Index: []

✅ Negative stock:
Empty DataFrame
Columns: [product_id, product_name, stock_qty]
Index: []


---
## 🌟 Bonus — Business Insights

### Top 3 Customers by Total Spend (Delivered orders only)

In [28]:
run("""
SELECT   c.customer_id,
         c.first_name || ' ' || c.last_name AS customer_name,
         COUNT(o.order_id)                  AS total_orders,
         SUM(o.total_amount)                AS total_spend
FROM     customers c
         JOIN orders o ON c.customer_id = o.customer_id
WHERE    o.status = 'Delivered'
GROUP BY c.customer_id, customer_name
ORDER BY total_spend DESC
LIMIT    3
""", "Bonus: Top 3 customers by spend")

▶ Bonus: Top 3 customers by spend


,customer_id,customer_name,total_orders,total_spend
0,101,Aarav Sharma,2,7997.0
1,105,Vikram Singh,1,5898.0
2,108,Divya Nair,1,1598.0


### Monthly Sales Trend

In [29]:
run("""
SELECT   strftime('%Y-%m', order_date) AS month,
         COUNT(*)                       AS order_count,
         SUM(total_amount)              AS monthly_revenue
FROM     orders
GROUP BY month
ORDER BY month
""", "Bonus: Monthly sales trend")
# Note: All sample data falls in August 2024

▶ Bonus: Monthly sales trend


,month,order_count,monthly_revenue
0,2024-08,10,35085.0


### Best-Selling Products by Quantity Sold

In [30]:
run("""
SELECT   p.product_name,
         SUM(oi.quantity) AS total_qty_sold
FROM     order_items oi
         JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_qty_sold DESC
LIMIT    5
""", "Bonus: Best-selling products")

▶ Bonus: Best-selling products


,product_name,total_qty_sold
0,Wireless Earbuds,3
1,Cushion Covers (Set),3
2,Smart Watch,2
3,Running Shoes,2
4,Laptop Stand,2


### Net Revenue Contribution by Category (after discounts)

In [31]:
run("""
SELECT   p.category,
         ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct/100)), 2) AS net_revenue
FROM     order_items oi
         JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY net_revenue DESC
""", "Bonus: Net revenue by category")
# Insight: Electronics drives ~64% of net item revenue (₹22,574.70)

▶ Bonus: Net revenue by category


,category,net_revenue
0,Electronics,19051.2
1,Clothing,9537.1
2,Home,4215.3


---
## 📝 Key Business Insights Summary

| # | Insight |
|---|---------|
| 1 | **60% fulfilment rate** — 6 of 10 orders Delivered; ₹13,596 still in-transit (Shipped) |
| 2 | **Top customer** — Aarav Sharma (₹7,997 across 2 delivered orders) |
| 3 | **Electronics dominates** revenue at ~64% of net item sales (₹22,575) |
| 4 | **Both Maharashtra customers are Premium** members — high-value segment |
| 5 | **Running Shoes & Bluetooth Speaker** are the highest per-item value products |
| 6 | **Home category** is budget-focused (avg ₹949); good volume driver |
| 7 | **No data quality issues** — zero NULLs, duplicates, orphaned records, or negative stock |